# 07 — Heat-conditioned accessibility

This notebook evaluates two consequences of the thermalized network:

1. **route-level change** — whether the generalized-cost shortest path differs from the time-shortest path;
2. **fixed-budget accessibility** — how many public-transport stop nodes are reachable within a 600 s generalized walking-cost budget.

The workflow also performs a sensitivity analysis over multiple values of \(\lambda\).

Important: \(\lambda\) is a sensitivity parameter in this PoC, not a behaviorally calibrated coefficient.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd

from shapely.geometry import box

THERMAL_GRAPH = DATA_INTERIM / "sevilla_walk_thermal.graphml"
ACCESS_OUTPUTS = OUTPUTS / "accessibility"
ACCESS_OUTPUTS.mkdir(parents=True, exist_ok=True)

TIME_BUDGET = 600
LAMBDA_VALUES = [0.00, 0.01, 0.025, 0.05, 0.075, 0.10]
RNG = np.random.default_rng(42)

G = ox.load_graphml(THERMAL_GRAPH)

numeric_fields = [
    "length",
    "walk_time_s",
    "utci_max",
    "heat_excess",
    "thermal_cost_s",
    "cost_ratio",
]

for _, _, _, data in G.edges(keys=True, data=True):
    for field in numeric_fields:
        if field in data:
            try:
                data[field] = float(data[field])
            except (TypeError, ValueError):
                data[field] = np.nan

## 1. Build the connected pre-clean graph

In [ ]:
largest_component = max(
    nx.strongly_connected_components(G),
    key=len,
)

G_preclean = G.subgraph(largest_component).copy()

print("Pre-clean nodes:", G_preclean.number_of_nodes())
print("Pre-clean edges:", G_preclean.number_of_edges())

## 2. Reconstruct transit-stop opportunities exactly as used in the PoC

In [ ]:
nodes_pre, edges_pre = ox.graph_to_gdfs(G_preclean)

thermal_polygon = gpd.GeoSeries(
    [box(*edges_pre.total_bounds)],
    crs=edges_pre.crs,
).to_crs(4326).iloc[0]

tags = {
    "highway": "bus_stop",
    "public_transport": ["platform", "stop_position"],
}

stops = ox.features_from_polygon(
    thermal_polygon,
    tags=tags,
)

stops = stops[
    stops.geometry.geom_type.isin(
        ["Point", "Polygon", "MultiPolygon"]
    )
].copy()

stops = stops.to_crs(G_preclean.graph["crs"])
stops["geometry"] = stops.geometry.centroid
stops = stops.drop_duplicates(subset="geometry")

stops["node"] = ox.distance.nearest_nodes(
    G_preclean,
    X=stops.geometry.x.values,
    Y=stops.geometry.y.values,
)

stop_nodes_preclean = list(pd.unique(stops["node"]))

print("Pre-clean unique stop nodes:", len(stop_nodes_preclean))

## 3. Remove edges with incomplete thermal information

In [ ]:
nodes_all, edges_all = ox.graph_to_gdfs(G_preclean)

edges_valid = edges_all[
    edges_all["walk_time_s"].notna()
    & edges_all["utci_max"].notna()
    & edges_all["heat_excess"].notna()
].copy()

valid_node_ids = (
    set(edges_valid.index.get_level_values("u"))
    | set(edges_valid.index.get_level_values("v"))
)

nodes_valid = nodes_all.loc[
    nodes_all.index.intersection(valid_node_ids)
].copy()

nodes_valid["x"] = nodes_valid.geometry.x
nodes_valid["y"] = nodes_valid.geometry.y

G_valid = ox.graph_from_gdfs(
    nodes_valid,
    edges_valid,
    graph_attrs={"crs": G_preclean.graph["crs"]},
)

largest_component = max(
    nx.strongly_connected_components(G_valid),
    key=len,
)

G_main = G_valid.subgraph(largest_component).copy()

stop_nodes = [
    node
    for node in stop_nodes_preclean
    if node in G_main
]

print("Final nodes:", G_main.number_of_nodes())
print("Final edges:", G_main.number_of_edges())
print("Retained stop nodes:", len(stop_nodes))

## 4. Add generalized costs for the sensitivity analysis

In [ ]:
for _, _, _, data in G_main.edges(keys=True, data=True):
    walk_time = float(data["walk_time_s"])
    heat_excess = float(data["heat_excess"])

    for lam in LAMBDA_VALUES:
        data[f"thermal_{lam}"] = (
            walk_time * (1 + lam * heat_excess)
        )

max_zero_difference = max(
    abs(float(data["thermal_0.0"]) - float(data["walk_time_s"]))
    for _, _, _, data in G_main.edges(keys=True, data=True)
)

print("Maximum λ=0 difference:", max_zero_difference)

## 5. Sample reproducible accessibility origins

In [ ]:
origin_nodes = RNG.choice(
    list(G_preclean.nodes),
    size=min(50, G_preclean.number_of_nodes()),
    replace=False,
)

# Cleaning can remove a small number of sampled nodes.
origin_nodes = [
    node for node in origin_nodes
    if node in G_main
]

print("Origins retained:", len(origin_nodes))

## 6. Accessibility sensitivity

In [ ]:
sensitivity_rows = []

for lam in LAMBDA_VALUES:
    weight = f"thermal_{lam}"

    for origin in origin_nodes:
        costs = nx.single_source_dijkstra_path_length(
            G_main,
            origin,
            cutoff=TIME_BUDGET,
            weight=weight,
        )

        reachable_stops = sum(
            stop in costs
            for stop in stop_nodes
        )

        sensitivity_rows.append({
            "lambda": lam,
            "origin": origin,
            "reachable_stops": reachable_stops,
        })

sensitivity = pd.DataFrame(sensitivity_rows)

sensitivity_summary = (
    sensitivity
    .groupby("lambda")["reachable_stops"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
    .reset_index()
)

baseline_mean = sensitivity_summary.loc[
    sensitivity_summary["lambda"] == 0.0,
    "mean",
].iloc[0]

sensitivity_summary["mean_access_loss_pct"] = (
    100
    * (baseline_mean - sensitivity_summary["mean"])
    / baseline_mean
)

display(sensitivity_summary)

## 7. Extract the main λ = 0.05 accessibility result

In [ ]:
baseline = (
    sensitivity.loc[
        sensitivity["lambda"] == 0.0,
        ["origin", "reachable_stops"],
    ]
    .rename(columns={"reachable_stops": "normal_access"})
)

heat_005 = (
    sensitivity.loc[
        sensitivity["lambda"] == 0.05,
        ["origin", "reachable_stops"],
    ]
    .rename(columns={"reachable_stops": "thermal_access"})
)

access = baseline.merge(
    heat_005,
    on="origin",
)

access["access_loss"] = (
    access["normal_access"]
    - access["thermal_access"]
)

access["access_loss_pct"] = np.where(
    access["normal_access"] > 0,
    100 * access["access_loss"] / access["normal_access"],
    np.nan,
)

display(access.describe())

## 8. Route-level comparison

In [ ]:
def route_metric(graph, route, attribute):
    total = 0.0

    for u, v in zip(route[:-1], route[1:]):
        values = [
            float(edge_data[attribute])
            for edge_data in graph[u][v].values()
            if attribute in edge_data
        ]

        if not values:
            return np.nan

        total += min(values)

    return total


def compare_routes(graph, origin, destination):
    fastest = nx.shortest_path(
        graph,
        origin,
        destination,
        weight="walk_time_s",
    )

    thermal = nx.shortest_path(
        graph,
        origin,
        destination,
        weight="thermal_cost_s",
    )

    fast_time = route_metric(graph, fastest, "walk_time_s")
    thermal_time = route_metric(graph, thermal, "walk_time_s")
    fast_thermal_cost = route_metric(graph, fastest, "thermal_cost_s")
    thermal_thermal_cost = route_metric(graph, thermal, "thermal_cost_s")

    return {
        "origin": origin,
        "destination": destination,
        "same_route": fastest == thermal,
        "fast_time_s": fast_time,
        "thermal_route_time_s": thermal_time,
        "fast_route_thermal_cost": fast_thermal_cost,
        "thermal_route_thermal_cost": thermal_thermal_cost,
        "extra_time_s": thermal_time - fast_time,
        "thermal_cost_reduction": fast_thermal_cost - thermal_thermal_cost,
        "thermal_cost_reduction_pct": (
            100
            * (fast_thermal_cost - thermal_thermal_cost)
            / fast_thermal_cost
            if fast_thermal_cost
            else np.nan
        ),
    }


node_array = np.array(list(G_main.nodes))
pairs = []

while len(pairs) < 100:
    origin, destination = RNG.choice(
        node_array,
        size=2,
        replace=False,
    )
    pairs.append((origin, destination))

route_rows = []

for origin, destination in pairs:
    try:
        route_rows.append(
            compare_routes(G_main, origin, destination)
        )
    except nx.NetworkXNoPath:
        pass

route_results = pd.DataFrame(route_rows)

changed_share = 100 * (~route_results["same_route"]).mean()
print(f"Route changed for {changed_share:.1f}% of sampled OD pairs.")

## 9. Save analysis outputs

In [ ]:
route_results.to_csv(
    ACCESS_OUTPUTS / "route_results_lambda_005.csv",
    index=False,
)

sensitivity_summary.to_csv(
    ACCESS_OUTPUTS / "accessibility_sensitivity.csv",
    index=False,
)

access.to_csv(
    ACCESS_OUTPUTS / "origin_accessibility_lambda_005.csv",
    index=False,
)

print("Saved accessibility outputs to:", ACCESS_OUTPUTS)

## Interpretation

The PoC demonstrates that embedding thermal stress in generalized walking cost can alter routes and reduce the number of transit opportunities reachable within a fixed budget. The magnitude depends strongly on the selected \(\lambda\), so the sensitivity analysis is essential.